# VisDrone ban ngày, 2 000 ảnh giao thông → Kaggle dataset

Notebook này **không train**. Nó lấy ảnh tĩnh của **VisDrone2019-DET** (Task 1: 10 209 ảnh drone chụp đô thị ở 14
thành phố Trung Quốc), chọn **2 000 ảnh giao thông ban ngày**, chép nguyên file JPEG gốc (không nén lại), rồi tạo
Kaggle dataset `<tài khoản>/visdrone-day-2000`. Bộ này là ảnh sạch, để sau này ghép lóe Wu et al. lên.

| | |
|---|---|
| nguồn | Kaggle `kushagrapandya/visdrone-dataset` (bản sao đủ 4 phần DET: train 6 471, val 548, test-dev 1 610, test-challenge 1 580 ảnh; đếm 20/10/2026). Gắn làm Input; không gắn thì Cell 4 tải bằng khoá Kaggle. Link Google Drive chính thức của VisDrone hay báo không tải được nên không dùng |
| ban ngày | độ sáng (luma) trung bình ≥ 90, phân vị 10% ≥ 25, ≤ 10% pixel tối (< 40), ≤ 2% pixel cháy sáng (> 250). Chỉ dùng độ sáng trung bình thì không đủ: ảnh đêm có đèn đường vẫn đạt 75–90 |
| giao thông | ≥ 5 xe (car, van, truck, bus) theo nhãn của VisDrone. `test-challenge` không có nhãn nên bị bỏ. Điều kiện này loại các cảnh như sân bóng rổ, công viên |
| đa dạng | ảnh cùng một chuỗi (số đầu tên file) rất giống nhau, nên chọn xoay vòng mỗi chuỗi một ảnh cho tới đủ 2 000; seed cố định |
| nội dung | `images/` (2 000 JPEG gốc, tên giữ nguyên), `index.csv` (phần gốc, chuỗi, cỡ, độ sáng, số xe), `README.md` |

Ngưỡng được dò trên toàn bộ 10 209 ảnh (xem ảnh thật ở từng khoảng độ sáng): 4 292 ảnh qua điều kiện ban ngày, 3 320
trong đó qua điều kiện xe, thuộc 225 chuỗi. Vẫn có thể lọt vài ảnh chụp lúc chạng vạng hoặc biển đèn neon: Cell 6 hiện
một lưới ảnh đã chọn để kiểm bằng mắt.

**Giấy phép.** Trang GitHub của VisDrone không ghi giấy phép. Bộ dữ liệu dùng cho nghiên cứu, yêu cầu trích dẫn Zhu et al.,
*Detection and Tracking Meet Drones Challenge*, TPAMI 2021. Vì vậy dataset mặc định **riêng tư**, license `other`.

## Chạy
1. *Add Input*: **`kushagrapandya/visdrone-dataset`** (khuyên dùng, khỏi tải 2,1 GB). Bản `banuprasadb/visdrone-dataset`
   (nhãn YOLO trong `labels/`) cũng dùng được.
2. *Settings*: **Accelerator: None**, **Internet: On**.
3. *Add-ons → Secrets*: **`KAGGLE_API_TOKEN`**, bật *Attached* cho notebook này. Không có secret thì notebook vẫn chạy
   hết (khi đã gắn Input): Cell 8 đóng gói vào `/kaggle/working`, Commit xong thì tạo dataset bằng *Output → New Dataset*.
4. *Save Version → Save & Run All (Commit)*. Ước tính 5–15 phút (chưa đo trên Kaggle).

## Cell 1 — Lựa chọn

In [ ]:
from pathlib import Path

SOURCE_DATASET = 'kushagrapandya/visdrone-dataset'
COUNT = 2000          # số ảnh chọn
SEED = 0
# Ban ngày: thống kê luma (0–255) trên ảnh thu 1/4 (draft JPEG)
DAY_MEAN = 90         # luma trung bình tối thiểu
DAY_P10 = 25          # phân vị 10% tối thiểu (ảnh đêm có đèn: trung bình cao nhưng nền rất tối)
DARK_LEVEL, DARK_MAX = 40, 0.10   # tỉ lệ pixel luma < 40 tối đa
CLIP_LEVEL, CLIP_MAX = 250, 0.02  # tỉ lệ pixel luma > 250 tối đa (ảnh cháy sáng)
# Giao thông: số hộp car(4), van(5), truck(6), bus(9) trong nhãn VisDrone tối thiểu
VEHICLE_CLASSES = {4, 5, 6, 9}
MIN_VEHICLES = 5
SPLITS = ('train', 'val', 'test-dev')  # test-challenge không có nhãn, nên không kiểm được điều kiện xe

SLUG = f'visdrone-day-{COUNT}'
TITLE = f'VisDrone daytime traffic {COUNT}'
SUBTITLE = f'{COUNT} daytime drone images of traffic from VisDrone2019-DET'
PUBLIC = False        # VisDrone không ghi giấy phép rõ: mặc định riêng tư
KAGGLE_OWNER = None   # None = tài khoản của khoá

DOWNLOAD = Path('/tmp/visdrone_download')  # chỉ dùng khi không gắn Input
UPLOAD = Path('/tmp/upload/visdrone_day')
OUTPUT_DIR = Path('/kaggle/working')
REPORT = OUTPUT_DIR / 'visdrone_day_report.md'

## Cell 2 — Hàm dùng chung (chỉ định nghĩa, chưa chạy gì)

In [ ]:
import csv
import json
import os
import posixpath
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor

import numpy as np
from PIL import Image


def gb(n):
    return f'{n / 1e9:.2f} GB'


def run(command, env=None, timeout=None):
    """(mã thoát, stdout, stderr) của một tiến trình con."""
    result = subprocess.run(command, env=env, capture_output=True, text=True, timeout=timeout)
    return result.returncode, result.stdout, result.stderr


def kaggle(*args, env, timeout=None):
    code, out, err = run([shutil.which('kaggle') or 'kaggle', *args], env=env, timeout=timeout)
    return code, (out + err).strip()


def find_splits(base, depth=5):
    """{'train': thư mục chứa images/ và annotations/, ...} của VisDrone2019-DET dưới ``base``."""
    found = {}
    for level in range(depth):
        for images in sorted(Path(base).glob('/'.join(['*'] * level + ['images']))):
            parent = images.parent
            if images.is_dir() and parent.name.startswith('VisDrone2019-DET-'):
                found.setdefault(parent.name.removeprefix('VisDrone2019-DET-'), parent)
    return found


def image_stats(path):
    """Thống kê luma trên ảnh thu 1/4 (draft JPEG, nhanh): (cỡ gốc, trung bình, p10, tỉ lệ tối, tỉ lệ cháy) hoặc lỗi."""
    try:
        with Image.open(path) as image:
            width, height = image.size
            image.draft('RGB', (width // 4, height // 4))
            pixels = np.asarray(image.convert('RGB'), dtype=np.float32)
        luma = 0.299 * pixels[..., 0] + 0.587 * pixels[..., 1] + 0.114 * pixels[..., 2]
        return {'width': width, 'height': height, 'mean': float(luma.mean()), 'p10': float(np.percentile(luma, 10)),
                'dark': float((luma < DARK_LEVEL).mean()), 'clip': float((luma > CLIP_LEVEL).mean()), 'error': None}
    except Exception as exc:
        return {'error': f'{type(exc).__name__}: {exc}'}


def label_file(root, stem):
    """Nhãn của một ảnh: annotations/<stem>.txt (VisDrone gốc, vd kushagrapandya/visdrone-dataset) hoặc
    labels/<stem>.txt (YOLO, vd banuprasadb/visdrone-dataset)."""
    for folder in ('annotations', 'labels'):
        if (root / folder).is_dir():
            return root / folder / f'{stem}.txt'
    return root / 'annotations' / f'{stem}.txt'


def vehicles(annotation):
    """Số hộp xe; None nếu không có nhãn. VisDrone gốc: 'x,y,w,h,score,category,...' với category trong
    VEHICLE_CLASSES. YOLO (cách đổi của Ultralytics, category − 1): 'class cx cy w h' với class + 1 trong VEHICLE_CLASSES."""
    if not annotation.is_file():
        return None
    count = 0
    for line in annotation.read_text(errors='replace').splitlines():
        line = line.strip()
        if ',' in line:
            values = line.rstrip(',').split(',')
            count += len(values) >= 6 and values[5].strip().isdigit() and int(values[5]) in VEHICLE_CLASSES
        elif line and line.split()[0].isdigit():
            count += int(line.split()[0]) + 1 in VEHICLE_CLASSES
    return count


def daytime_reason(row):
    """None nếu ảnh qua điều kiện ban ngày, không thì lý do."""
    if row['mean'] < DAY_MEAN:
        return f'tối (trung bình {row["mean"]:.0f} < {DAY_MEAN})'
    if row['p10'] < DAY_P10:
        return f'nền tối (p10 {row["p10"]:.0f} < {DAY_P10})'
    if row['dark'] > DARK_MAX:
        return f'nhiều pixel tối ({row["dark"]:.0%})'
    if row['clip'] > CLIP_MAX:
        return f'cháy sáng ({row["clip"]:.0%})'
    return None


def pick_round_robin(rows, count, seed):
    """Xoay vòng các chuỗi (số đầu tên file), mỗi vòng lấy một ảnh ngẫu nhiên của mỗi chuỗi, tới đủ ``count``."""
    rng = np.random.default_rng(seed)
    groups = defaultdict(list)
    for row in sorted(rows, key=lambda row: row['file']):
        groups[row['sequence']].append(row)
    queues = {key: [group[i] for i in rng.permutation(len(group))] for key, group in sorted(groups.items())}
    order = list(queues)
    picked = []
    while len(picked) < count and any(queues.values()):
        for key in [order[i] for i in rng.permutation(len(order))]:
            if queues[key] and len(picked) < count:
                picked.append(queues[key].pop())
    return picked


# Chạy trong tiến trình con có ENV: liệt kê mọi file của dataset qua API, mỗi dòng một JSON.
LIST_FILES = r"""
import json, sys
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()
token = None
while True:
    response = api.dataset_list_files(sys.argv[1], token, int(sys.argv[2]))
    if response is None or getattr(response, "error_message", None):
        print(json.dumps({"error": getattr(response, "error_message", None) or "no response"}))
        break
    for item in response.files or []:
        print(json.dumps({"name": item.name, "bytes": getattr(item, "total_bytes", None)}))
    token = response.next_page_token
    if not token:
        break
"""


def remote_files(dataset, env, page_size=200):
    """[{'name', 'bytes'}] của phiên bản mới nhất của dataset."""
    code, out, err = run([sys.executable, '-c', LIST_FILES, dataset, str(page_size)], env=env, timeout=1800)
    rows = [json.loads(line) for line in out.splitlines() if line.startswith('{')]
    errors = [row['error'] for row in rows if 'error' in row]
    if code != 0 or errors:
        raise RuntimeError(f'liệt kê file thất bại (mã {code}): {errors or err.strip()[-1500:]}')
    return rows


USAGE = """import csv
from pathlib import Path
from PIL import Image

# Attached datasets are mounted under /kaggle/input/<dataset> (newer notebooks may use
# /kaggle/input/datasets/<owner>/<dataset>), so look for the folder that holds index.csv.
ROOT = next(path.parent for depth in range(0, 5)
            for path in sorted(Path('/kaggle/input').glob('/'.join(['*'] * depth + ['index.csv'])))
            if (path.parent / 'images').is_dir())
rows = list(csv.DictReader(open(ROOT / 'index.csv')))
# Hold out whole sequences: the VisDrone test-dev images, whose sequences are not in train or val.
train = [ROOT / 'images' / row['file'] for row in rows if row['source_split'] != 'test-dev']
valid = [ROOT / 'images' / row['file'] for row in rows if row['source_split'] == 'test-dev']
image = Image.open(train[0]).convert('RGB')
"""
print('đã định nghĩa hàm')

## Cell 3 — Môi trường và khoá Kaggle

Cài `kaggle>=2.2`, đọc secret, thử đăng nhập. Khoá không được in ra. Không có khoá vẫn chạy được khi đã gắn Input.

In [ ]:
import importlib.metadata

code_, out, err = run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', 'kaggle>=2.2'])
assert code_ == 0, f'pip install lỗi:\n{err[-2000:]}'
print('kaggle', importlib.metadata.version('kaggle'))
print(run(['df', '-h', '/tmp', '/kaggle/working'])[1])

try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
except ImportError:  # ngoài Kaggle: đọc biến môi trường
    _secrets = None
secret_errors = {}


def secret(name):
    try:
        return _secrets.get_secret(name) if _secrets else os.environ.get(name)
    except Exception as error:
        # thông báo không chứa giá trị secret; hay gặp nhất: secret chưa tạo hoặc chưa Attached cho notebook này
        secret_errors[name] = f'{type(error).__name__}: {str(error)[:300]}'
        return None


_token = secret('KAGGLE_API_TOKEN')
_key = None if _token else secret('KAGGLE_KEY')
OWNER = KAGGLE_OWNER or (None if _token else secret('KAGGLE_USERNAME'))
UPLOAD_BY_API = bool(_token or (_key and OWNER))
ENV, DATASET, EXISTS = None, None, False
if not UPLOAD_BY_API:
    for name, error in secret_errors.items():
        print(f'{name}: {error}')
    print('\nKHÔNG CÓ KHOÁ KAGGLE: cần gắn Input kushagrapandya/visdrone-dataset; Cell 8 đóng gói dataset vào '
          '/kaggle/working, Commit xong thì tạo dataset bằng tab Output → New Dataset.')
else:
    # Khoá chỉ nằm trong ENV, truyền cho tiến trình con kaggle. Bỏ khoá Kaggle có sẵn trong môi trường: token cũ được
    # ưu tiên hơn cặp USERNAME/KEY, sẽ đăng nhập nhầm tài khoản.
    ENV = {name: value for name, value in os.environ.items()
           if name not in ('KAGGLE_API_TOKEN', 'KAGGLE_USERNAME', 'KAGGLE_KEY')}
    ENV.update({'KAGGLE_API_TOKEN': _token} if _token else {'KAGGLE_USERNAME': OWNER, 'KAGGLE_KEY': _key})
del _key, _token
if UPLOAD_BY_API and not OWNER:  # tên tài khoản của token, hỏi qua API
    code_, out, err = run([sys.executable, '-c', 'from kaggle.api.kaggle_api_extended import KaggleApi; api = KaggleApi(); '
                           'api.authenticate(); print(api.get_config_value("username") or "")'], env=ENV, timeout=120)
    OWNER = (out.strip().splitlines() or [''])[-1].strip()
    if code_ != 0 or not OWNER:
        raise SystemExit(f'KAGGLE_API_TOKEN không đăng nhập được (token sai hoặc hết hạn?): {err.strip()[-500:]}')
if UPLOAD_BY_API:
    code_, text = kaggle('datasets', 'list', '--mine', '--page-size', '1', env=ENV)
    if code_ != 0 or any(word in text.lower() for word in ('401', '403', 'unauthorized', 'forbidden')):
        raise SystemExit(f'Lệnh kaggle chưa đăng nhập được (mã {code_}): {text[-500:]}')
    DATASET = f'{OWNER}/{SLUG}'
    code_, text = kaggle('datasets', 'status', DATASET, env=ENV)
    EXISTS = code_ == 0 and not any(word in text.lower() for word in ('404', 'not found'))
    print(f'tài khoản Kaggle: {OWNER} | {DATASET}: '
          f'{"đã có, Cell 8 sẽ thêm version mới" if EXISTS else "chưa có, Cell 8 sẽ tạo mới"}')

## Cell 4 — Tìm VisDrone2019-DET

Tìm trong `/kaggle/input` các thư mục `VisDrone2019-DET-<phần>/images`. Không thấy thì tải `kushagrapandya/visdrone-dataset`
bằng khoá Kaggle vào `/tmp` (2,1 GB) và giải nén.

In [ ]:
ROOTS = find_splits('/kaggle/input')
if not all(split in ROOTS for split in SPLITS):
    if not UPLOAD_BY_API:
        raise SystemExit(f'Không thấy VisDrone2019-DET trong /kaggle/input (thấy {sorted(ROOTS)}) và không có khoá để '
                         f'tải. Add Input → {SOURCE_DATASET}, rồi chạy lại.')
    print(f'Không thấy trong Input: tải {SOURCE_DATASET} vào {DOWNLOAD}')
    DOWNLOAD.mkdir(parents=True, exist_ok=True)
    code_, text = kaggle('datasets', 'download', SOURCE_DATASET, '-p', str(DOWNLOAD), env=ENV, timeout=3 * 3600)
    print(text[-1000:])
    archives = sorted(DOWNLOAD.glob('*.zip'))
    assert code_ == 0 and archives, f'Tải {SOURCE_DATASET} thất bại (mã {code_})'
    for archive in archives:
        with zipfile.ZipFile(archive) as handle:
            handle.extractall(DOWNLOAD)
        archive.unlink()
    ROOTS = find_splits(DOWNLOAD)
missing = [split for split in SPLITS if split not in ROOTS]
assert not missing, f'thiếu phần {missing} (thấy {sorted(ROOTS)})'
for split, root in sorted(ROOTS.items()):
    images = sum(1 for _ in (root / 'images').glob('*.jpg'))
    labels = sum(1 for _ in label_file(root, 'x').parent.glob('*.txt'))
    print(f'{split:<15} {images:>5} ảnh {labels:>5} nhãn  {root}{"" if split in SPLITS else "  (không dùng)"}')

## Cell 5 — Đo độ sáng, đếm xe, chọn ảnh

Mọi ảnh của các phần dùng được đo độ sáng trên bản thu 1/4 và đếm xe trong nhãn. Ảnh qua cả hai điều kiện được chọn
xoay vòng theo chuỗi tới đủ `COUNT`. Không đủ ảnh thì báo **vấn đề** (dataset riêng tư) và lấy hết số có.

In [ ]:
PROBLEMS, WARNINGS = [], []
paths = [(split, path) for split in SPLITS for path in sorted((ROOTS[split] / 'images').glob('*.jpg'))]
started = time.perf_counter()
with ProcessPoolExecutor(max_workers=os.cpu_count() or 4) as pool:
    stats = list(pool.map(image_stats, [str(path) for _, path in paths], chunksize=64))
ROWS, reasons = [], Counter()
for (split, path), stat in zip(paths, stats):
    if stat['error']:
        WARNINGS.append(f'{split}/{path.name} không đọc được: {stat["error"]}')
        reasons['không đọc được'] += 1
        continue
    row = {'file': path.name, 'source_split': split, 'sequence': path.name.split('_')[0], **stat,
           'vehicles': vehicles(label_file(ROOTS[split], path.stem)), 'path': path}
    row['reason'] = daytime_reason(row) or (
        'không có nhãn' if row['vehicles'] is None else
        f'ít xe ({row["vehicles"]} < {MIN_VEHICLES})' if row['vehicles'] < MIN_VEHICLES else None)
    reasons[(row['reason'] or 'qua').split(' (')[0]] += 1
    ROWS.append(row)
print(f'đo {len(paths)} ảnh trong {time.perf_counter() - started:.0f} s')
for reason, count in reasons.most_common():
    print(f'  {reason:<20} {count:>6}')
names = Counter(row['file'] for row in ROWS)
assert all(count == 1 for count in names.values()), 'trùng tên file giữa các phần: images/ phẳng sẽ ghi đè'

CANDIDATES = [row for row in ROWS if row['reason'] is None]
SELECTED = sorted(pick_round_robin(CANDIDATES, COUNT, SEED), key=lambda row: row['file'])
if len(SELECTED) < COUNT:
    PROBLEMS.append(f'chỉ {len(SELECTED)} ảnh qua điều kiện, cần {COUNT}')
per_sequence = Counter(row['sequence'] for row in SELECTED)
print(f'\nqua điều kiện: {len(CANDIDATES)} ảnh, {len({row["sequence"] for row in CANDIDATES})} chuỗi')
print(f'đã chọn: {len(SELECTED)} ảnh, {len(per_sequence)} chuỗi, nhiều nhất {max(per_sequence.values())} ảnh/chuỗi')
print('theo phần gốc:', dict(Counter(row['source_split'] for row in SELECTED)))
splits_of = defaultdict(set)
for row in SELECTED:
    splits_of[row['sequence']].add(row['source_split'])
SHARED = Counter('+'.join(sorted(splits)) for splits in splits_of.values() if len(splits) > 1)
TESTDEV_ALONE = not any('test-dev' in key for key in SHARED)
print('chuỗi nằm ở nhiều phần gốc:', dict(SHARED) or 'không có',
      '| test-dev không chung chuỗi nào với phần khác' if TESTDEV_ALONE else '| test-dev CHUNG chuỗi với phần khác')
print('cỡ ảnh:', ', '.join(f'{w}×{h} ({n})' for (w, h), n in
                          Counter((row['width'], row['height']) for row in SELECTED).most_common(6)))
for problem in PROBLEMS:
    print('VẤN ĐỀ', problem)

## Cell 6 — Xem bằng mắt

Hàng trên: 18 ảnh đã chọn ngẫu nhiên. Hàng dưới: 6 ảnh bị loại vì "nền tối" hoặc "nhiều pixel tối" dù trung bình
≥ 90, tức là ảnh sát ngưỡng. Thấy ảnh đêm lọt vào ảnh đã chọn thì tăng `DAY_MEAN`/`DAY_P10` ở Cell 1 rồi chạy lại từ Cell 5.

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(SEED + 1)
show = [SELECTED[i] for i in rng.choice(len(SELECTED), min(18, len(SELECTED)), replace=False)]
near = [row for row in ROWS if row['reason'] and row['mean'] >= DAY_MEAN and row['reason'].startswith(('nền', 'nhiều'))]
show_near = [near[i] for i in rng.choice(len(near), min(6, len(near)), replace=False)] if near else []
figure, axes = plt.subplots(4, 6, figsize=(18, 10))
for axis, row in zip(axes.flat, show + [None] * (18 - len(show)) + show_near):
    axis.axis('off')
    if row is None:
        continue
    with Image.open(row['path']) as image:
        image.draft('RGB', (row['width'] // 4, row['height'] // 4))
        axis.imshow(image.convert('RGB'))
    label = 'LOẠI: ' + row['reason'] if row['reason'] else f'{row["source_split"]} | {row["vehicles"]} xe'
    axis.set_title(f'{row["file"][:24]}\nm{row["mean"]:.0f} p10 {row["p10"]:.0f} | {label}', fontsize=7)
plt.tight_layout()
plt.show()

## Cell 7 — Thư mục upload

Chép nguyên từng byte 2 000 file JPEG đã chọn vào `/tmp/upload/visdrone_day/images/`, ghi `index.csv`, `README.md`
(tiếng Anh, cũng là mô tả trên Kaggle) và `dataset-metadata.json` (CLI chỉ đọc, không upload thành file).

In [ ]:
if UPLOAD.exists():
    shutil.rmtree(UPLOAD)
(UPLOAD / 'images').mkdir(parents=True)
for row in SELECTED:
    shutil.copyfile(row['path'], UPLOAD / 'images' / row['file'])
FIELDS = ['file', 'source_split', 'sequence', 'width', 'height', 'mean', 'p10', 'dark', 'clip', 'vehicles']
with open(UPLOAD / 'index.csv', 'w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=FIELDS, extrasaction='ignore')
    writer.writeheader()
    for row in SELECTED:
        writer.writerow({key: round(value, 4) if isinstance(value, float) else value for key, value in row.items()})
FINAL = sorted(path.name for path in (UPLOAD / 'images').iterdir())
assert FINAL == [row['file'] for row in SELECTED], 'images/ khác danh sách đã chọn'
assert all((UPLOAD / 'images' / row['file']).stat().st_size == row['path'].stat().st_size for row in SELECTED)

split_counts = Counter(row['source_split'] for row in SELECTED)
README = f"""# {TITLE}

{len(SELECTED)} daytime drone images of urban traffic, selected from the still-image set of **VisDrone2019-DET**
(Task 1). Files are the original JPEGs, byte for byte, with their original names. They are meant as clean images to
composite lens flare onto.

**Source and terms.** VisDrone (AISKYEYE team, Tianjin University): https://github.com/VisDrone/VisDrone-Dataset .
Taken from the Kaggle mirror `{SOURCE_DATASET}`. The VisDrone page states no license; use it for research and cite
P. Zhu, L. Wen, D. Du, X. Bian, H. Fan, Q. Hu, H. Ling, "Detection and Tracking Meet Drones Challenge", IEEE TPAMI,
2021. This is an unofficial subset; the uploader is not an author.

## Selection

- **Daytime**, measured on luma (0–255) of a 1/4-size decode: mean ≥ {DAY_MEAN}, 10th percentile ≥ {DAY_P10},
  at most {DARK_MAX:.0%} of pixels below {DARK_LEVEL}, at most {CLIP_MAX:.0%} of pixels above {CLIP_LEVEL}.
- **Traffic**: at least {MIN_VEHICLES} boxes of car, van, truck or bus in the VisDrone annotation. The
  test-challenge part has no annotations and was not used.
- **Diversity**: images of one sequence (first field of the file name) are near duplicates, so images were taken
  round-robin, one per sequence per round (seed {SEED}): {len(per_sequence)} sequences, at most
  {max(per_sequence.values())} images per sequence.
- {len(CANDIDATES)} of {len(ROWS)} images passed both tests.

## Contents

- `images/`: {len(SELECTED)} JPEG images ({', '.join(f'{count} from {split}' for split, count in sorted(split_counts.items()))}).
- `index.csv`: `file, source_split, sequence, width, height, mean, p10, dark, clip, vehicles`. To hold out
  validation images, split by `sequence`{" or use `source_split == 'test-dev'` (no test-dev sequence appears in train or val)" if TESTDEV_ALONE else " (here test-dev shares sequences with other parts)"}.

## Usage

```python
{USAGE.rstrip()}
```
"""
(UPLOAD / 'README.md').write_text(README, encoding='utf-8')
assert 6 <= len(TITLE) <= 50 and 20 <= len(SUBTITLE) <= 80, 'Kaggle: title 6–50 ký tự, subtitle 20–80 ký tự'
VISIBLE_PUBLIC = PUBLIC and not PROBLEMS
if UPLOAD_BY_API:
    (UPLOAD / 'dataset-metadata.json').write_text(json.dumps(
        {'title': TITLE, 'id': DATASET, 'licenses': [{'name': 'other'}], 'subtitle': SUBTITLE,
         'description': README}, indent=2, ensure_ascii=False))
total = sum(path.stat().st_size for path in UPLOAD.rglob('*') if path.is_file())
print(f'{UPLOAD}: {len(FINAL)} ảnh + index.csv, README.md | {gb(total)} | dataset sẽ',
      'PUBLIC' if VISIBLE_PUBLIC else 'RIÊNG TƯ')

## Cell 8 — Upload

**Có khoá:** `kaggle datasets create -p /tmp/upload/visdrone_day --dir-mode zip`; dataset đã có thì thêm version mới.
CLI có lúc in lỗi mà vẫn trả mã 0, nên cell đọc cả chữ trong output.

**Không có khoá:** `images/` thành `images.tar` không nén trong `/kaggle/working` (*New Dataset* từ Output chỉ nhận
500 file đầu), kèm `index.csv`, `README.md`. Kaggle giải nén `images.tar` thành `images/`.

In [ ]:
import tarfile
import tempfile

PACKED = 0
if UPLOAD_BY_API:
    free = shutil.disk_usage(tempfile.gettempdir()).free
    assert free > total * 1.05 + 1e9, f'{tempfile.gettempdir()} còn {gb(free)}, zip tạm cần {gb(total)}'
    if EXISTS:
        args = ['datasets', 'version', '-p', str(UPLOAD), '--dir-mode', 'zip', '-m', f'{len(SELECTED)} images, seed {SEED}']
        print('Dataset đã có: thêm version mới (chế độ hiển thị giữ như cũ)')
    else:
        args = ['datasets', 'create', '-p', str(UPLOAD), '--dir-mode', 'zip'] + (['--public'] if VISIBLE_PUBLIC else [])
    started = time.perf_counter()
    code_, text = kaggle(*args, env=ENV, timeout=4 * 3600)
    print(text[-4000:])
    print(f'{time.perf_counter() - started:.0f} s')
    failed = [word for word in ('error', 'unsuccessful', 'skipping') if word in text.lower()]
    assert code_ == 0 and not failed, f'Upload thất bại (mã {code_}, thấy {failed}); xem output ở trên'
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    free = shutil.disk_usage(OUTPUT_DIR).free
    assert free > total + 1e9, f'{OUTPUT_DIR} còn {gb(free)}, cần {gb(total)}'
    target, partial = OUTPUT_DIR / 'images.tar', OUTPUT_DIR / 'images.tar.part'
    written = {}
    with tarfile.open(partial, 'w', format=tarfile.PAX_FORMAT) as tar:
        for name in FINAL:
            tar.add(UPLOAD / 'images' / name, arcname=name, recursive=False)
            written[name] = (UPLOAD / 'images' / name).stat().st_size
    with tarfile.open(partial) as tar:
        read_back = {member.name: member.size for member in tar if member.isfile()}
    assert read_back == written, f'{partial}: tar đọc lại khác lúc ghi'
    os.replace(partial, target)
    PACKED = len(written)
    for name in ('index.csv', 'README.md'):
        shutil.copy2(UPLOAD / name, OUTPUT_DIR / name)
    print(f"""{target.name}: {PACKED} file, {gb(target.stat().st_size)}, đọc lại khớp

Commit xong:
  1. Mở trang phiên bản vừa Commit → tab Output → New Dataset.
  2. Tên: {SLUG}. Chọn images.tar, index.csv, README.md.
  3. Đặt {"Public" if VISIBLE_PUBLIC else "Private"}; dán README.md làm mô tả.
  4. Chờ Kaggle giải nén: trang dataset phải có images/ ({PACKED} file).""")

## Cell 9 — Xác minh trên Kaggle, dọn dẹp, báo cáo

Chờ trạng thái `ready`, liệt kê file của dataset qua API, so số file với lúc upload, rồi xoá dữ liệu trong `/tmp` và
biến chứa khoá. Báo cáo ghi vào `/kaggle/working/visdrone_day_report.md` (chỉ khi upload bằng khoá).

In [ ]:
status, started = ('', time.perf_counter()) if UPLOAD_BY_API else ('chưa tạo: tạo tay từ tab Output', 0)
for _ in range(120 if UPLOAD_BY_API else 0):  # Kaggle giải nén các zip sau khi nhận xong; chờ tối đa 60 phút
    code_, text = kaggle('datasets', 'status', DATASET, env=ENV)
    status = (text.splitlines() or [''])[-1].strip().lower()
    if status in ('ready', 'error'):
        break
    time.sleep(30)
REMOTE = remote_files(DATASET, ENV) if status == 'ready' else []
remote_counts = Counter(posixpath.dirname(row['name']) or '.' for row in REMOTE)
local_counts = Counter({'images': len(FINAL), '.': 2})  # index.csv, README.md
VERIFY = [(folder, local_counts[folder], remote_counts[folder] if REMOTE else '-')
          for folder in sorted(set(local_counts) | set(remote_counts))]
MISMATCHED = [row for row in VERIFY if REMOTE and row[1] != row[2]]
if REMOTE:
    print(f'{"thư mục":<10} {"local":>7} {"Kaggle":>7}')
    for folder, here, there in VERIFY:
        print(f'{folder:<10} {here:>7} {there:>7}{"" if here == there else "   <- LỆCH"}')

shutil.rmtree(UPLOAD.parent, ignore_errors=True)
shutil.rmtree(DOWNLOAD, ignore_errors=True)
del ENV  # khoá chỉ từng nằm trong biến này

if UPLOAD_BY_API:
    link = f'https://www.kaggle.com/datasets/{DATASET}'
    visibility = 'như version trước (dataset đã có)' if EXISTS else 'public' if VISIBLE_PUBLIC else 'riêng tư'
else:
    link = f'chưa tạo: tab Output → New Dataset, tên {SLUG}'
    visibility = 'nên đặt public' if VISIBLE_PUBLIC else 'nên giữ riêng tư'
lines = [
    '# VisDrone ban ngày → Kaggle: báo cáo', '',
    f'- Dataset: {link} ({visibility}); trạng thái Kaggle: {status}',
    f'- Nguồn: {SOURCE_DATASET}, phần {", ".join(SPLITS)}',
    f'- Qua điều kiện: {len(CANDIDATES)}/{len(ROWS)} ảnh; đã chọn {len(SELECTED)} ảnh, {len(per_sequence)} chuỗi',
    f'- Theo phần gốc: {dict(split_counts)}', '',
    '## Lý do loại', '',
] + [f'- {reason}: {count}' for reason, count in reasons.most_common()]
lines += ['', '## Vấn đề', ''] + ([f'- {problem}' for problem in PROBLEMS] or ['- không có'])
lines += ['', '## Cảnh báo', ''] + ([f'- {warning}' for warning in WARNINGS[:20]] or ['- không có'])
lines += ['', '## Đối chiếu sau upload', '']
lines += [f'- `{folder}`: local {here}, Kaggle {there}' for folder, here, there in MISMATCHED] or (
    ['- khớp từng thư mục'] if REMOTE else ['- chưa liệt kê được file trên Kaggle' if UPLOAD_BY_API else
                                         '- bỏ qua: dataset tạo tay từ Output, kiểm trên trang dataset'])
lines += ['', '## Đọc dữ liệu', '', '```python', USAGE.rstrip(), '```', '']
REPORT_TEXT = '\n'.join(lines)
if UPLOAD_BY_API:
    REPORT.parent.mkdir(parents=True, exist_ok=True)
    REPORT.write_text(REPORT_TEXT, encoding='utf-8')
print(REPORT_TEXT)
assert not MISMATCHED, f'{len(MISMATCHED)} thư mục lệch số file giữa local và Kaggle'